# Dataset explorer — every planted shock at a glance

Minimal companion to the three generated datasets. Each section lists what was
planted (from the config — **spoilers**, don't hand this to analysts) and shows
one figure where it is visible. This notebook reads the raw event log + ground
truth, so regenerate with `--raw`:

```bash
python main.py generate --config configs/notion_easy.yaml --seed 42 --raw
python main.py generate --config configs/notion_hard.yaml --seed 42 --raw --hide-truth
python main.py generate --config configs/chess_medium.yaml --seed 42 --raw
```

For the full dashboards use `python main.py analyze output/<name> --save`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from analysis import load, retention_table

causal = load(Path("output/notion_easy"))
sneaky = load(Path("output/notion_hard"))
chess  = load(Path("output/chess_medium"))


def shade(ax, shocks, colors=("red", "orange", "purple", "brown")):
    for color, s in zip(colors, shocks):
        ax.axvspan(s["start_week"], s["end_week"], color=color, alpha=0.12,
                   label=s.get("description"))


def by_type(weekly):
    return (weekly.groupby(["abs_week", "event_type"])["event_count"]
            .sum().unstack(fill_value=0))


def wau_by_segment(d):
    act = d["weekly"][["user_id", "abs_week"]].drop_duplicates()
    act = act.merge(d["users"][["user_id", "segment"]], on="user_id")
    return act.groupby(["abs_week", "segment"])["user_id"].nunique().unstack()


## 1. `notion_easy` — easy: three obvious step changes

| Shock | Window | Where to see it |
|---|---|---|
| Marketing paused (acquisition ×0.4) | weeks 10–14 | new signups dip (panel 1) |
| AI server crash (`used_ai` ×0) | weeks 25–28 | `used_ai` flatlines **and** `page_view` dips with it — page views are causal (panel 2) |
| Competitor launch (retention ×0.7) | weeks 40–52 | weekly active users break trend (panel 3) |


In [ ]:
shocks = causal["config"]["shocks"]
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

signups = causal["users"].groupby("cohort_week").size()
axes[0].plot(signups.index, signups.values, color="#1f77b4")
shade(axes[0], shocks)
axes[0].set_title("New signups / week")

bt = by_type(causal["weekly"])
for col in bt.columns:
    if bt[col].loc[5] >= 50:  # skip rare types
        axes[1].plot(bt.index, bt[col] / bt[col].loc[5] * 100, label=col, lw=1.4)
shade(axes[1], shocks)
axes[1].set_title("Events by type (week 5 = 100)")
axes[1].legend(fontsize=7)

wau = causal["weekly"].groupby("abs_week")["user_id"].nunique()
axes[2].plot(wau.index, wau.values, color="black")
shade(axes[2], shocks)
axes[2].set_title("Weekly active users")

axes[0].legend(fontsize=7)
for ax in axes:
    ax.set_xlabel("week"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 2. `notion_hard` — hard: every anomaly's obvious diagnosis is wrong

| Shock | Window | Trap → truth |
|---|---|---|
| Tracking bug: 65% of `commented` lost | weeks 12–16 | looks like an engagement drop, but `page_view` (true behaviour) doesn't move → instrumentation, not product (panel 1) |
| Price increase: casual retention ×0.8, survivors' activity ×1.3 | weeks 30+ | topline barely moves (0.8 × 1.3 ≈ 1) — only the segment cut shows casual WAU breaking (panel 3); the downgrade wave is the root-cause clue (panel 2) |
| Onboarding regression: cohorts signing up on release 2.4 retain ×0.85 *for life* | cohorts 20+ | blended retention looks like slow drift; the cohort cut shows a sharp break at week 20 (panel 4) |

Two A/B tests also run on top (honest `share_button_redesign`, novelty-trapped
`ai_assist_prompt`) — see `experiments.png` from `main.py analyze` for those.


In [ ]:
cfg = sneaky["config"]
bt = by_type(sneaky["weekly"])
fig, axes = plt.subplots(1, 4, figsize=(22, 4))

for col in ("commented", "page_view"):
    axes[0].plot(bt.index, bt[col] / bt[col].loc[5] * 100, label=col, lw=1.6)
shade(axes[0], [s for s in cfg["shocks"] if s["type"] == "instrumentation"])
axes[0].set_title("Tracking bug: commented drops, page views don't")
axes[0].legend(fontsize=7)

axes[1].plot(bt.index, bt["downgraded"], color="#d62728")
shade(axes[1], [s for s in cfg["shocks"] if s["type"] == "transition"])
axes[1].set_title("Downgrade wave = price-increase clue")

seg_wau = wau_by_segment(sneaky)
for seg, color in (("casual", "gray"), ("core", "#1f77b4")):
    axes[2].plot(seg_wau.index, seg_wau[seg], label=seg, color=color, lw=1.6)
axes[2].axvline(30, color="red", ls="--", lw=1, label="price increase (wk 30)")
axes[2].set_title("WAU by segment: only casual breaks")
axes[2].legend(fontsize=7)

ret4 = retention_table(sneaky["weekly"])[4].dropna()
axes[3].plot(ret4.index, ret4.values, color="purple", marker="o", ms=3)
axes[3].axvline(20, color="black", ls="--", lw=1, label="release 2.4 (wk 20)")
axes[3].set_title("Week-4 retention by signup cohort")
axes[3].legend(fontsize=7)

for ax in axes:
    ax.set_xlabel("week"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 3. `chess_medium` — single event: the structure *is* the time series

One event type (`game_played`), no feature mix to dissect. Planted instead:

- **Winter seasonality** — acquisition/activity peak in January, trough mid-year.
- **AR(1) noise** — autocorrelated weekly wobble; good and bad weeks cluster.
- **World Championship spike (week 46)** — one-off acquisition ×1.8.
- **Duolingo lifecycle retention** (CURR/RURR/SURR…) instead of decay curves.

Panel 3 plots the *realized* weekly multipliers from
`ground_truth_timeseries.csv` — the answer key the first two panels should echo.


In [ ]:
spike = chess["config"]["time_series"]["spikes"][0]
ts = pd.read_csv("output/chess_medium/ground_truth_timeseries.csv")
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

games = chess["weekly"].groupby("abs_week")["event_count"].sum()
axes[0].plot(games.index, games.values, color="#2b6f3f")
axes[0].set_title("Games played / week")

signups = chess["users"].groupby("cohort_week").size()
axes[1].plot(signups.index, signups.values, color="#1f77b4")
axes[1].axvline(spike["week"], color="red", ls="--", lw=1, label=spike["description"])
axes[1].set_title("New players / week")
axes[1].legend(fontsize=7)

for col in ("acquisition_mult", "activity_mult", "retention_mult"):
    axes[2].plot(ts["week"], ts[col], label=col, lw=1.4)
axes[2].axhline(1.0, color="gray", lw=0.8)
axes[2].set_title("Ground truth: realized weekly multipliers")
axes[2].legend(fontsize=7)

for ax in axes:
    ax.set_xlabel("week"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
